In [ ]:
import numpy as np
m = np.memmap("inews.7622", "h")

In [ ]:
n = 4
a,b = np.unique(np.lib.stride_tricks.as_strided(m[:0].view(f'V{2*n+2}'), [len(m)-n], [2]), return_counts=1)
a,c = np.unique(a.view(f'V{2*n},H')['f0'], return_inverse=1)
a = np.zeros(len(a), 'I')
np.maximum.at(a, c, b)
a.sum() / len(m)

In [ ]:
n = 4
a,b = np.unique(np.lib.stride_tricks.as_strided(m[:0].view(f'V{2*n+2}'), [len(m)-n], [2]), return_counts=1)
a.view('4H,H')['f0'].sort()
a,c = np.unique(a, return_inverse=1)
d = np.zeros(len(a), 'I')
np.add.at(d, c, b)
b = d
a,c = np.unique(a.view(f'V{2*n},H')['f0'], return_inverse=1)
a = np.zeros(len(a), 'I')
np.maximum.at(a, c, b)
a.sum() / len(m)

In [ ]:
import numpy as np
m = np.memmap("../nlpdat/py/all.B", mode='r')

In [ ]:
n = 4
a,b = np.unique(np.lib.stride_tricks.as_strided(m[:0].view(f'V{n+1}'), [len(m)-n], [1]), return_counts=1)
a,c = np.unique(a.view(f'V{n},B')['f0'], return_inverse=1)
a = np.zeros(len(a), 'I')
np.maximum.at(a, c, b)
a.sum() / len(m)

In [ ]:
n = 8
a,b = np.unique(np.lib.stride_tricks.as_strided(m[:0].view(f'V{n+1}'), [len(m)-n], [1]), return_counts=1)
a,c = np.unique(a.view(f'V{n},B')['f0'], return_inverse=1)
a = np.zeros(len(a), 'I')
np.maximum.at(a, c, b)
a.sum() / len(m)

In [ ]:
n = 12
a,b = np.unique(np.lib.stride_tricks.as_strided(m[:0].view(f'V{n+1}'), [len(m)-n], [1]), return_counts=1)
a,c = np.unique(a.view(f'V{n},B')['f0'], return_inverse=1)
a = np.zeros(len(a), 'I')
np.maximum.at(a, c, b)
a.sum() / len(m)

In [ ]:
%reset -f in out

In [ ]:
import numba as nb

In [ ]:
nb.b

In [ ]:
from numba import njit
import numpy as np
# @njit('u4[:](u1[:,:],i8[:])')
# def cat(a, s):
#     r = np.empty(len(s), 'I')
#     r[s[0]] = 0
#     t = 0
#     for i in range(1, len(s)):
#         t += (a[s[i-1]] != a[s[i]]).any()
#         r[s[i]] = t
#     return r
@njit('u4[:](u1[:],i8[:])')
def cat(a, s):
    w = len(a) - len(s) + 1
    t = 0
    o = np.empty(len(s), 'I')
    o[s[0]] = 0
    for i in range(1, len(s)):
        t += (a[s[i-1]:s[i-1]+w] != a[s[i]:s[i]+w]).any()
        o[s[i]] = t
    return o

In [ ]:
import numpy as np
m = np.memmap("../nlpdat/py/all.B", mode='r')

In [ ]:
from collections import defaultdict, Counter
class Id(int):
    __slots__ = ()
    def __hash__(s):
        return hash(mS[s])
    def __eq__(s, S):
        return mS[s] == mS[S]
n = 8
mS = np.lib.stride_tricks.as_strided(m[:0].view(f'S{n}'), [len(m)-n], [1])
a = defaultdict(Counter)
for i,j in enumerate(m[n:]):
    a[Id(i)][j] += 1
sum(a.most_common(1)[0][1]for a in a.values()) / len(m)

In [ ]:
a = []

In [ ]:
a.index(4)

In [ ]:
from collections import defaultdict
class Id(int):
    __slots__ = ()
    def __hash__(s):
        return hash(mS[s])
    def __eq__(s, S):
        return mS[s] == mS[S]
class Count(list):
    __slots__ = 'v',
    def __init__(s):
        s.v = []
    def __call__(s, a):
        try:
            s.v[s.index(a)] += 1
        except:
            s.append(a)
            s.v.append(1)
n = 8
mS = np.lib.stride_tricks.as_strided(m[:0].view(f'S{n}'), [len(m)-n], [1])
a = defaultdict(Count)
for i,j in enumerate(m[n:]):
    a[Id(i)](j)
sum(max(a.v)for a in a.values()) / len(m)

In [ ]:
from numba import njit
import numpy as np
@njit('b1(u1[:],u1[:])', inline='always')
def gt(a, b):
    for i in range(len(a)):
        if a[i] > b[i]:
            return True
        elif a[i] < b[i]:
            return False
    return False
@njit('void(u4[:],u1[:,:])', inline='always')
def qsort(a, m):
    i = 0
    j = len(a) - 1
    p = m[a[len(a)//2]]
    while True:
        while gt(p, m[a[i]]):
            i += 1
        while gt(m[a[j]], p):
            j -= 1
        if i <= j:
            t = a[i]
            a[i] = a[j]
            a[j] = t
            i += 1
            j -= 1
        else:
            break
    if j:
        qsort(a[:j+1], m)
    if i < len(a) - 1:
        qsort(a[i:], m)
@njit('u4(u1[:,:],u4[:])')
def acc(a, s):
    o = 0
    N = n = 1
    for i in range(1, len(s)):
        if (a[s[i-1],:-1] == a[s[i],:-1]).all():
            if a[s[i-1],-1] == a[s[i],-1]:
                n += 1
            else:
                N = max(N, n)
                n = 1
        else:
            o += N
            N = 1
    return o + max(N, n)

In [ ]:
import numpy as np
m = np.memmap("../nlpdat/py/all.B")

In [ ]:
s = np.arange(len(m)-32, dtype='I')
qsort(s, np.lib.stride_tricks.sliding_window_view(m, 33, writeable=True))

In [ ]:
s.tofile('../nlpdat/py/all.sort.I')

In [ ]:
a = np.lib.stride_tricks.sliding_window_view(m, 9, writeable=True)
s = np.arange(len(a), dtype='I')
qsort(s, a)
acc(a, s)

In [ ]:
acc(np.lib.stride_tricks.as_strided(m, [len(s),5], [1,1]), s) / len(m)

In [ ]:
acc(np.lib.stride_tricks.as_strided(m, [len(s),9], [1,1]), s) / len(m)

In [ ]:
acc(np.lib.stride_tricks.as_strided(m, [len(s),17], [1,1]), s) / len(m)

In [ ]:
acc(np.lib.stride_tricks.as_strided(m, [len(s),33], [1,1]), s) / len(m)

In [ ]:
3|4

In [ ]:
import numpy as np
m = np.full(1000000, 255, 'B')
m[[*range(ord('a'),ord('z')+1),*range(ord('A'),ord('Z')+1),*b'0123456789_.+-*@/%^<>~&|=()[]{}, \n\t"\'\\']] = range(90)
f = open('all.B', 'wb')

In [ ]:
%reset -f in out

In [ ]:
a = m[np.array([open('../nlpdat/py/ipynb.txt', encoding='utf8').read()]).view('I')]
f.write(a[a<255])

In [ ]:
a = m[np.array([open('../nlpdat/py/ipynbp.txt', encoding='utf8').read()]).view('I')]
f.write(a[a<255])

In [ ]:
a = m[np.array([open('../nlpdat/py/py.txt', encoding='utf8').read()]).view('I')]
f.write(a[a<255])

In [ ]:
a = m[np.array([open('../nlpdat/py/pyp.txt', encoding='utf8').read()]).view('I')]
f.write(a[a<255])

In [ ]:
f.close()

In [ ]:
f = open('all.B', 'wb')
a = np.array([open('../nlpdat/py/ipynb.txt', encoding='utf8').read()]).view('I')
f.write(a.view('4B')[a<127,0])

In [ ]:
a = np.array([open('../nlpdat/py/ipynbp.txt', encoding='utf8').read()]).view('I')
f.write(a.view('4B')[a<127,0])

In [ ]:
a = np.array([open('../nlpdat/py/py.txt', encoding='utf8').read()]).view('I')
f.write(a.view('4B')[a<127,0])

In [ ]:
a = np.array([open('../nlpdat/py/pyp.txt', encoding='utf8').read()]).view('I')
f.write(a.view('4B')[a<127,0])

In [ ]:
a

In [ ]:
s

In [ ]:
a[s]

In [ ]:
import numpy as np
m = np.memmap("../nlpdat/py/all.B", mode='r')

In [ ]:
chr(127)

In [ ]:
len(m)*8/2**30

In [ ]:
class Id(int):
    __slots__ = ()
    def __hash__(s):
        return hash(mS[s])
    def __eq__(s, S):
        return mS[s] == mS[S]
mS = np.lib.stride_tricks.as_strided(m[:0].view(f'S{33}'), [len(m)-32], [1])

In [ ]:
s = [Id(i) for i in range(len(a))]

In [ ]:
s = np.lib.stride_tricks.as_strided(m[:0].view(f'S{33}'), [len(m)-32], [1]).argsort()

In [ ]:
n = 12
mS = np.lib.stride_tricks.as_strided(m[:0].view(f'S{n}'), [len(m)-n], [1])
a = defaultdict(Count)
for i,j in enumerate(m[n:]):
    a[Id(i)](j)
sum(max(a.v)for a in a.values()) / len(m)

In [ ]:
import numpy as np
m = np.memmap("../nlpdat/py/all.90", mode='r')

In [ ]:
a = np.zeros(90, 'I')
np.add.at(a, m, 1)
np.log2(len(m)) - a @ np.log2(a) / len(m)

In [ ]:
a = np.zeros(90**2, 'I')
np.add.at(a, np.lib.stride_tricks.sliding_window_view(m, 2) @ (90,1), 1)
a = a[a.astype('?')]
np.log2(len(m)) - a @ np.log2(a) / len(m)

In [ ]:
a = np.zeros(90**4, 'I')
np.add.at(a, np.lib.stride_tricks.sliding_window_view(m, 4) @ (90**3,90**2,90,1), 1)
a = a[a.astype('?')]
np.log2(len(m)) - a @ np.log2(a) / len(m)

In [ ]:
4.743542475623531*8

In [ ]:
8.461105812721279*4

In [ ]:
13.401341019798815*2

In [ ]:
4.743542475623531*8+8.461105812721279*4+13.401341019798815*2+16